# Step 12a -- CARE-Sim Non-Causal Evaluation

Evaluate the trained CARE-Sim non-causal model on the broad ICU simulator problem.

This notebook assumes you already uploaded:
- `caresim_colab.zip`
- `data/rl_dataset_noncausal.parquet`
- `models/icu_readmit/caresim_noncausal/`

It will:
1. mount Drive and verify the GPU
2. unzip the source bundle
3. run a short smoke evaluation
4. run the full non-causal `12a` evaluation when you are ready

If you only want a quick check that the model is functional, Cell 3 is enough.


In [ ]:
# Cell 1: Mount Drive + check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Full evaluation will be slower on CPU.')


In [ ]:
# Cell 2: Unzip source files + verify inputs
import os, sys, zipfile

DRIVE_ROOT = '/content/drive/MyDrive/CareAI'
ZIP_CANDIDATES = [
    os.path.join(DRIVE_ROOT, 'caresim_colab.zip'),
    '/content/drive/MyDrive/caresim_colab.zip',
]
UNZIP_DIR = '/content/caresim_src'
SRC_PATH = os.path.join(UNZIP_DIR, 'src')
DATA_PATH = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_noncausal.parquet')
MODEL_DIR = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'caresim_noncausal')
SMOKE_REPORT_DIR = os.path.join(DRIVE_ROOT, 'reports', 'icu_readmit', 'caresim_noncausal_smoke')
FULL_REPORT_DIR = os.path.join(DRIVE_ROOT, 'reports', 'icu_readmit', 'caresim_noncausal')
EVAL_SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_12a_caresim_evaluate_noncausal.py')

ZIP_PATH = next((p for p in ZIP_CANDIDATES if os.path.exists(p)), None)
if ZIP_PATH is None:
    raise FileNotFoundError(
        'caresim_colab.zip not found. Upload it to MyDrive/CareAI/ or MyDrive/.\n'
        'Run python scripts/prepare_colab_upload.py locally first.'
    )

print(f'Unzipping {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(UNZIP_DIR)
print(f'Unzipped to {UNZIP_DIR}')

for pkg_dir in [
    os.path.join(SRC_PATH, 'careai'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit', 'caresim'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit', 'caresim', 'control'),
]:
    init = os.path.join(pkg_dir, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Data file not found:\n  {DATA_PATH}\n'
        'Upload rl_dataset_noncausal.parquet to MyDrive/CareAI/data/'
    )
if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(
        f'Model dir not found:\n  {MODEL_DIR}\n'
        'Upload the trained caresim_noncausal folder to MyDrive/CareAI/models/icu_readmit/'
    )
if not os.path.exists(EVAL_SCRIPT):
    raise FileNotFoundError(f'Evaluation script not found: {EVAL_SCRIPT}')

print(f'Data file OK:  {DATA_PATH}')
print(f'Model dir OK:  {MODEL_DIR}')
print(f'Smoke reports: {SMOKE_REPORT_DIR}')
print(f'Full reports:  {FULL_REPORT_DIR}')
print('Ready.')


In [ ]:
# Cell 3: Smoke evaluation (quick sanity check)
import subprocess

def run_streaming(cmd, env):
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

SMOKE_ARGS = [
    sys.executable, '-u', EVAL_SCRIPT,
    '--data', DATA_PATH,
    '--model-dir', MODEL_DIR,
    '--report-dir', SMOKE_REPORT_DIR,
    '--device', device,
    '--smoke',
]

rc = run_streaming(SMOKE_ARGS, env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'})
if rc != 0:
    raise RuntimeError('Non-causal CARE-Sim smoke evaluation FAILED.')
print('\nNon-causal CARE-Sim smoke evaluation complete.')


In [ ]:
# Cell 4: Full 12a evaluation
FULL_ARGS = [
    sys.executable, '-u', EVAL_SCRIPT,
    '--data', DATA_PATH,
    '--model-dir', MODEL_DIR,
    '--report-dir', FULL_REPORT_DIR,
    '--device', device,
]

print('=' * 60)
print('FULL EVALUATION -- non-causal CARE-Sim')
print('=' * 60)
rc = run_streaming(FULL_ARGS, env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'})
if rc != 0:
    raise RuntimeError('Non-causal CARE-Sim full evaluation FAILED.')
print('\nNon-causal CARE-Sim full evaluation complete.')


In [ ]:
# Cell 5: Inspect report outputs
for label, root in [('smoke', SMOKE_REPORT_DIR), ('full', FULL_REPORT_DIR)]:
    print(f'\n[{label}] {root}')
    if not os.path.exists(root):
        print('  (missing)')
        continue
    for current_root, dirs, files in os.walk(root):
        for f in sorted(files):
            path = os.path.join(current_root, f)
            size_kb = os.path.getsize(path) / 1024
            rel = os.path.relpath(path, root)
            print(f'  {rel:55s}  {size_kb:8.1f} KB')

print('\nIf you only wanted a quick sanity check, Cell 3 is enough.')
